# Stage 3. Participant timeline

Audience: whoever is diagnosing one specific problem. Answers *what actually
happened to this person*.

Six lanes on a shared 00:00-24:00 local clock, so events line up vertically -
the point being that a wear gap under a flat sync trace is a data-delivery
failure, while the same gap under a live sync trace is genuine non-wear.

Deliberately absent: EMA response latency and the 30-minute in-window flag.
`sent_at` and `responded_at` are stamped from the same `now` at submit time, so
every latency would read as ~0 and every row as in-window - a flawless,
meaningless compliance curve.

In [1]:
# Every notebook under backend/dashboard/stages/ shares one data layer: the Django
# bootstrap, the SYNTHETIC_DATA switch, the fourteen frames and every compute
# function. Set SYNTHETIC_DATA in monitor_common.py to swap fixture for ORM.
from monitor_common import *

print("data source:", describe_source())

data source: Django ORM - c5dqhursbgn9fb.cluster-czrs8kj4isg7.us-east-1.rds.amazonaws.com / dd28mps21fsprq


In [2]:
# STAGE 3A — SIX-LANE DAILY STRIP
# Lane 1 is minute-resolution: wear_lane bins to the minute, which is what
# wear_coverage scores against, and the fixture stores one sample per minute.
LANE_TITLES = ("Wear", "Sync", "Check-ins", "Decision points", "Delivery", "MSSD")


def _ref(kind, row):
    """Plotly names the first subplot axis 'x'/'y', not 'x1'/'y1'."""
    return kind if row == 1 else f"{kind}{row}"


def _waking_band(fig, row):
    """08:00-22:00 in every lane, not just wear: it is the denominator the day
    is scored against, and shading it everywhere is what makes the vertical
    alignment between lanes traceable."""
    start, end = WAKING_WINDOW_START_HOUR * 60, WAKING_WINDOW_END_HOUR * 60
    fig.add_shape(type="rect", x0=start, x1=end, y0=0, y1=1,
                  xref=_ref("x", row), yref=_ref("y", row) + " domain",
                  fillcolor=ink("band"), opacity=0.45, line=dict(width=0), layer="below")


def _clock_axis(fig, row, last):
    fig.update_xaxes(range=[0, MINUTES_PER_DAY], tickvals=CLOCK_TICKS,
                     ticktext=CLOCK_LABELS if last else [""] * len(CLOCK_TICKS),
                     showgrid=True, gridcolor=ink("grid"), row=row, col=1)


def _lane_wear(fig, lane, row):
    start, end = lane["waking_window"]
    fig.add_shape(type="rect", x0=start, x1=end, y0=0, y1=1,
                  xref=_ref("x", row), yref=_ref("y", row) + " domain",
                  fillcolor=ink("band"), opacity=1.0, line=dict(width=0),
                  layer="below")
    for edge in (start, end):
        fig.add_shape(type="line", x0=edge, x1=edge, y0=0, y1=1,
                      xref=_ref("x", row), yref=_ref("y", row) + " domain",
                      line=dict(color=ink("axis"), width=1, dash="dot"))
    fig.add_annotation(x=start, y=1.0, xref=_ref("x", row), yref=_ref("y", row) + " domain",
                       xanchor="left", yanchor="bottom", text="waking window 08:00-22:00",
                       showarrow=False, font=dict(size=9, color=ink("muted")))
    if not lane["has_data"]:
        fig.add_annotation(x=MINUTES_PER_DAY / 2, y=0.5, xref=_ref("x", row), yref=_ref("y", row) + " domain",
                           text="no heart-rate samples", showarrow=False,
                           font=dict(size=10, color=ink("muted")))
        return
    bins = lane["bins"]
    fig.add_trace(go.Heatmap(
        z=[[1 if value else 0 for value in bins]],
        x=list(range(MINUTES_PER_DAY)), y=["worn"],
        colorscale=[[0.0, "rgba(0,0,0,0)"], [1.0, ink("series_1")]],
        showscale=False, hovertemplate="minute %{x}<extra></extra>"), row=row, col=1)
    for gap in lane["gaps_gt_2h"]:
        clipped_start, clipped_end = max(gap["start"], start), min(gap["end"], end)
        if clipped_end - clipped_start <= 0:
            continue
        fig.add_shape(type="rect", x0=clipped_start, x1=clipped_end, y0=0, y1=1,
                      xref=_ref("x", row), yref=_ref("y", row) + " domain",
                      fillcolor=STATUS["critical"], opacity=0.28, line=dict(width=0))
        fig.add_annotation(x=(clipped_start + clipped_end) / 2, y=0.82,
                           xref=_ref("x", row), yref=_ref("y", row) + " domain",
                           text=f"{clipped_end - clipped_start} min", showarrow=False,
                           font=dict(size=9, color=STATUS["critical"]))


def _lane_sync(fig, lane, local_date, row):
    if not lane["advances"] and lane["carried_in"] is None:
        message = ("no sync writer - unmeasurable" if not lane["measurable"]
                   else "no sync advance recorded")
        fig.add_annotation(x=MINUTES_PER_DAY / 2, y=0.5, xref=_ref("x", row), yref=_ref("y", row) + " domain",
                           text=message, showarrow=False,
                           font=dict(size=10, color=ink("muted")))
        return
    minutes, lags = [], []
    for advance in lane["advances"]:
        observed = advance["minute"]
        synced = local_minutes(advance["last_synced_at"], local_date)
        if observed is None:
            continue
        minutes.append(observed)
        lags.append(None if synced is None else max(0.0, observed - synced))
    fig.add_trace(go.Scatter(
        x=minutes, y=lags, mode="lines+markers", line=dict(color=ink("series_1"), width=2,
                                                           shape="hv"),
        marker=dict(size=8, color=ink("series_1"), line=dict(color=ink("surface"), width=2)),
        hovertemplate="sync at %{x:.0f} min<br>clock %{y:.0f} min behind<extra></extra>",
        showlegend=False), row=row, col=1)


COMPLETENESS_BANDS = (
    ("complete", 1.0, "series_1"),
    ("partial", 0.0, "series_4"),
    ("missing B1/B2", None, "critical"),
)


def checkin_band(mark):
    """Three states, not a continuous ramp. The third is the one that matters:
    a check-in without B1/B2 produces no MSSD and therefore no decision point."""
    if mark.get("missing_b1b2"):
        return "missing B1/B2"
    value = mark.get("completeness")
    if value is not None and value >= 1.0:
        return "complete"
    return "partial"


def _lane_checkins(fig, lane, row):
    if not lane["emas"] and not lane["reminders"]:
        fig.add_annotation(x=MINUTES_PER_DAY / 2, y=0.5, xref=_ref("x", row),
                           yref=_ref("y", row) + " domain",
                           text="no check-in submitted and no reminder sent", showarrow=False,
                           font=dict(size=10, color=ink("muted")))
    for slot in lane["slots"]:
        fig.add_shape(type="line", x0=slot["start"], x1=slot["start"], y0=0, y1=1,
                      yref=_ref("y", row) + " domain", line=dict(color=ink("grid"), width=1, dash="dash"))
    for band, _value, role in COMPLETENESS_BANDS:
        marks = [mark for mark in lane["emas"] if checkin_band(mark) == band]
        if not marks:
            continue
        colour = STATUS["critical"] if role == "critical" else ink(role)
        fig.add_trace(go.Scatter(
            x=[mark["minute"] for mark in marks],
            y=[mark["ema_type"] for mark in marks], mode="markers",
            name=f"check-in: {band}", legendgroup=band,
            marker=dict(size=12, color=colour, line=dict(color=ink("surface"), width=2)),
            customdata=[[mark["ema_id"],
                         "-" if mark["completeness"] is None else f"{mark['completeness']:.0%}",
                         "yes" if mark["missing_b1b2"] else "no"] for mark in marks],
            hovertemplate=("EMA %{customdata[0]} (%{y})<br>completeness %{customdata[1]}"
                           "<br>missing B1/B2: %{customdata[2]}<extra></extra>"),
            showlegend=True), row=row, col=1)
    if lane["reminders"]:
        fig.add_trace(go.Scatter(
            x=[tick["minute"] for tick in lane["reminders"]],
            y=["reminder"] * len(lane["reminders"]), mode="markers+text",
            marker=dict(symbol="line-ns", size=10, line=dict(color=ink("series_2"), width=2)),
            text=[str(tick["daily_count_at_send"]) for tick in lane["reminders"]],
            textposition="middle right", textfont=dict(size=9, color=ink("muted")),
            hovertemplate="reminder, slot index %{text}<extra></extra>",
            showlegend=False), row=row, col=1)


def _lane_decisions(fig, points, row):
    hues = dict(zip(DECISION_ROWS, series_hues(len(DECISION_ROWS))))
    for outcome in DECISION_ROWS:
        marks = [point for point in points if point["outcome"] == outcome]
        fig.add_trace(go.Scatter(
            x=[mark["minute"] for mark in marks],
            y=[DECISION_LABEL[outcome]] * len(marks), mode="markers",
            marker=dict(size=11, color=hues[outcome],
                        line=dict(color=ink("surface"), width=2)),
            customdata=[[mark["decision_point_id"],
                         "-" if mark["observed_mssd"] is None else round(mark["observed_mssd"], 3),
                         "-" if mark["randomization_draw"] is None
                         else round(mark["randomization_draw"], 3)] for mark in marks],
            hovertemplate=("%{customdata[0]}<br>MSSD %{customdata[1]}"
                           "<br>draw %{customdata[2]}<extra></extra>"),
            showlegend=False), row=row, col=1)
    fig.add_trace(go.Scatter(
        x=[None] * len(DECISION_ROWS),
        y=[DECISION_LABEL[outcome] for outcome in DECISION_ROWS],
        mode="markers", marker=dict(size=0.1, color="rgba(0,0,0,0)"),
        hoverinfo="skip", showlegend=False), row=row, col=1)
    other = [point for point in points if point["outcome"] not in DECISION_ROWS]
    if other:
        fig.add_trace(go.Scatter(
            x=[mark["minute"] for mark in other], y=["other"] * len(other), mode="markers",
            marker=dict(size=11, color=ink("muted"), line=dict(color=ink("surface"), width=2)),
            hovertemplate="%{text}<extra></extra>",
            text=[mark["outcome"] for mark in other], showlegend=False), row=row, col=1)


def _lane_delivery(fig, prompts, row):
    if not prompts:
        fig.add_annotation(x=MINUTES_PER_DAY / 2, y=0.5, xref=_ref("x", row), yref=_ref("y", row) + " domain",
                           text="no prompt sent", showarrow=False,
                           font=dict(size=10, color=ink("muted")))
        return
    for index, prompt in enumerate(prompts):
        label = f"#{prompt['jitai_log_id']}"
        window = prompt["outcome_window"]
        if window[0] is not None and window[1] is not None:
            fig.add_shape(type="rect", x0=window[0], x1=window[1], y0=index - 0.35,
                          y1=index + 0.35, fillcolor=ink("series_1"), opacity=0.12,
                          line=dict(width=0), layer="below")
        span = [value for value in (prompt["push_sent_minute"],
                                    prompt["device_received_minute"],
                                    prompt["receipt_reported_minute"]) if value is not None]
        if span:
            fig.add_trace(go.Scatter(
                x=[min(span), max(span) + 1], y=[label, label], mode="lines",
                line=dict(color=ink("series_1"), width=8),
                hovertemplate=(f"{label}<br>{prompt['delivery_status']}"
                               f"<br>{prompt['receipt_platform'] or 'unknown'} /"
                               f" {prompt['receipt_app_state'] or '-'}<extra></extra>"),
                showlegend=False), row=row, col=1)
        if prompt["delivery_status"] == "failed":
            fig.add_trace(go.Scatter(
                x=[prompt["push_sent_minute"]], y=[label], mode="markers",
                marker=dict(symbol="x", size=11, color=STATUS["critical"]),
                hovertemplate=f"{prompt['delivery_error']}<extra></extra>",
                showlegend=False), row=row, col=1)
        if prompt["linked_ema_responded_minute"] is not None:
            fig.add_trace(go.Scatter(
                x=[prompt["linked_ema_responded_minute"]], y=[label], mode="markers",
                marker=dict(symbol="diamond", size=10, color=ink("series_3"),
                            line=dict(color=ink("surface"), width=2)),
                hovertemplate="post-prompt check-in<extra></extra>",
                showlegend=False), row=row, col=1)


def _lane_mssd(fig, points, row):
    if not points:
        fig.add_annotation(x=MINUTES_PER_DAY / 2, y=0.5, xref=_ref("x", row), yref=_ref("y", row) + " domain",
                           text="no decision point", showarrow=False,
                           font=dict(size=10, color=ink("muted")))
        return
    ordered = sorted(points, key=lambda point: point["minute"] or 0)
    fig.add_trace(go.Scatter(
        x=[point["minute"] for point in ordered],
        y=[point["observed_mssd"] for point in ordered], mode="lines+markers",
        line=dict(color=ink("series_1"), width=2, shape="hv"),
        marker=dict(size=8, color=ink("series_1"), line=dict(color=ink("surface"), width=2)),
        name="observed MSSD",
        hovertemplate="observed %{y:.3f}<extra></extra>", showlegend=False), row=row, col=1)
    fig.add_trace(go.Scatter(
        x=[point["minute"] for point in ordered],
        y=[point["threshold"] for point in ordered], mode="lines",
        line=dict(color=ink("muted"), width=2, dash="dot", shape="hv"),
        name="threshold", hovertemplate="threshold %{y:.3f}<extra></extra>",
        showlegend=False), row=row, col=1)
    flagged = [point for point in ordered if point["unexplained"]]
    if flagged:
        fig.add_trace(go.Scatter(
            x=[point["minute"] for point in flagged],
            y=[point["observed_mssd"] for point in flagged], mode="markers",
            name="over threshold, no draw taken",
            marker=dict(symbol="star", size=16, color=STATUS["critical"],
                        line=dict(color=ink("surface"), width=2)),
            hovertemplate="over threshold but no draw taken<extra></extra>",
            showlegend=True), row=row, col=1)
        for point in flagged:
            fig.add_annotation(
                x=point["minute"], y=point["observed_mssd"],
                xref=_ref("x", row), yref=_ref("y", row),
                text="engine cleared the threshold but never drew", showarrow=True,
                arrowhead=0, arrowcolor=STATUS["critical"], arrowwidth=1,
                ax=0, ay=-26, font=dict(size=9, color=STATUS["critical"]))


def plot_timeline_day(user_id, local_date):
    day = {
        "wear": wear_lane(user_id, local_date),
        "sync": sync_lane(user_id, local_date),
        "checkins": checkin_lane(user_id, local_date),
        "decisions": decision_lane(user_id, local_date),
        "delivery": delivery_lane(user_id, local_date),
        "mssd": mssd_lane(user_id, local_date),
    }
    fig = make_subplots(rows=6, cols=1, shared_xaxes=True, vertical_spacing=0.035,
                        row_heights=[0.10, 0.13, 0.19, 0.20, 0.20, 0.18],
                        subplot_titles=LANE_TITLES)
    _lane_wear(fig, day["wear"], 1)
    _lane_sync(fig, day["sync"], local_date, 2)
    _lane_checkins(fig, day["checkins"], 3)
    _lane_decisions(fig, day["decisions"], 4)
    _lane_delivery(fig, day["delivery"], 5)
    _lane_mssd(fig, day["mssd"], 6)

    for row in range(1, 7):
        if row != 1:
            _waking_band(fig, row)
        _clock_axis(fig, row, last=(row == 6))
        fig.update_yaxes(showgrid=False, tickfont=dict(size=10, color=ink("muted")),
                         row=row, col=1)
    fig.update_yaxes(showticklabels=False,
                     title=dict(text="worn", font=dict(size=9, color=ink("muted"))),
                     row=1, col=1)
    fig.update_yaxes(title=dict(text="decision outcome",
                                font=dict(size=9, color=ink("muted"))), row=4, col=1)
    fig.update_yaxes(title=dict(text="prompt", font=dict(size=9, color=ink("muted"))),
                     row=5, col=1)
    fig.update_yaxes(title=dict(text="MSSD (unitless)",
                                font=dict(size=9, color=ink("muted"))), row=6, col=1)
    fig.update_xaxes(title=dict(text="local clock time (America/New_York)",
                                font=dict(size=11, color=ink("muted"))), row=6, col=1)
    for row, key in ((3, "checkins"), (4, "decisions"), (5, "delivery"), (6, "mssd")):
        empty = not day[key] if key != "checkins" else not (
            day["checkins"]["emas"] or day["checkins"]["reminders"])
        if empty:
            fig.update_yaxes(showticklabels=False, row=row, col=1)
    fig.update_yaxes(title=dict(text="sync clock lag (min)",
                                font=dict(size=9, color=ink("muted"))),
                     rangemode="tozero", row=2, col=1)
    fig.update_yaxes(categoryorder="array",
                     categoryarray=[DECISION_LABEL[name] for name in DECISION_ROWS] + ["other"],
                     row=4, col=1)

    study_day = study_day_for_local(user_id, local_date)
    base_layout(fig, 940,
                f"Participant {user_id} - {local_date} (study day {study_day})",
                margin=dict(l=140, r=40, t=112, b=64),
                context={"n": 1})
    # Stage 3 exists to answer "what else was happening at 14:20", so the hover
    # reads every lane at one x rather than one mark at a time.
    fig.update_layout(plot_bgcolor=ink("surface"), hovermode="x unified",
                      showlegend=True,
                      legend=dict(orientation="h", y=1.03, x=0, yanchor="bottom",
                                  font=dict(size=10, color=ink("text_secondary"))))
    fig.update_xaxes(showspikes=True, spikemode="across", spikesnap="cursor",
                     spikecolor=ink("axis"), spikethickness=1, spikedash="dot")
    for annotation in fig.layout.annotations[:len(LANE_TITLES)]:
        annotation.update(x=0, xanchor="left", font=dict(size=11, color=ink("text")))
    return fig


def study_day_for_local(user_id, local_date):
    anchor = day1_dates().get(user_id)
    return None if anchor is None else (local_date - anchor).days


def plot_timeline(user_id, days=7, end=None):
    end = end or today_local()
    return [plot_timeline_day(user_id, end - timedelta(days=offset))
            for offset in reversed(range(days))]

In [3]:
# STAGE 3B — DELIVERY FUNNEL
def plot_delivery_funnel(user_id):
    payload = delivery_funnel(user_id)
    stages = payload["stages"]
    fig = go.Figure(go.Funnel(
        orientation="h", y=list(stages["stage"]), x=list(stages["n"]),
        marker=dict(color=ink("series_1"), line=dict(color=ink("surface"), width=2)),
        connector=dict(line=dict(color=ink("axis"), width=1, dash="dot")),
        text=[f"<b>{int(n)}</b>" + ("" if pd.isna(pct) or not pct else f"  -{pct:.0%}")
              for n, pct in zip(stages["n"], stages["drop_pct"])],
        textposition="inside", textfont=dict(size=12, color=ink("surface")),
        textinfo="text",
        hovertemplate="%{y}<br>%{x} prompts<extra></extra>"))
    for index, row in enumerate(stages.to_dict("records")):
        if not row["drop_from_previous"] or pd.isna(row["drop_from_previous"]):
            continue
        fig.add_annotation(
            x=1.0, xref="paper", xanchor="left", y=row["stage"], xshift=6, showarrow=False,
            text=(f"<span style='color:{STATUS['critical']}'>"
                  f"-{int(row['drop_from_previous'])}</span>"),
            font=dict(size=11))
    base_layout(fig, 360, f"Delivery funnel - participant {user_id}",
                margin=dict(l=170, r=80, t=80, b=60))
    fig.update_xaxes(title=dict(text="prompts (count)",
                                font=dict(size=11, color=ink("muted"))))
    return fig


def plot_delivery_splits(user_id):
    """Platform and app state as two charts. They measure different things, so
    one chart with two scales would invent a relationship that is not there."""
    payload = delivery_funnel(user_id)
    figures = {}
    for column, title in (("receipt_platform", "Delivered by platform"),
                          ("receipt_app_state", "Delivered by app state")):
        counts = payload["splits"].get(column) or {}
        fig = go.Figure()
        if counts:
            keys = list(counts)
            fig.add_trace(go.Bar(
                x=keys, y=[counts[key] for key in keys],
                marker=dict(color=series_hues(max(len(keys), 1))[:len(keys)],
                            line=dict(color=ink("surface"), width=2)),
                text=[counts[key] for key in keys], textposition="outside",
                textfont=dict(size=11, color=ink("text")),
                hovertemplate="%{x}<br>%{y} delivered<extra></extra>", showlegend=False))
        else:
            fig.add_annotation(x=0.5, y=0.5, xref="paper", yref="paper", showarrow=False,
                               text="nothing delivered", font=dict(size=11, color=ink("muted")))
        base_layout(fig, 270, title, margin=dict(l=70, r=40, t=70, b=60))
        fig.update_yaxes(rangemode="tozero", showgrid=True, gridcolor=ink("grid"),
                         title=dict(text="delivered prompts (count)",
                                    font=dict(size=11, color=ink("muted"))))
        fig.update_xaxes(title=dict(text=column.replace("receipt_", "").replace("_", " "),
                                    font=dict(size=11, color=ink("muted"))))
        figures[column] = fig
    return figures

In [4]:
# STAGE 3C — ITEM COMPLETENESS MATRIX
def plot_completeness_matrix(user_id, local_date):
    matrix = completeness_matrix(user_id, local_date)
    fig = go.Figure()
    if matrix.empty:
        fig.add_annotation(x=0.5, y=0.5, xref="paper", yref="paper", showarrow=False,
                           text="no check-in submitted that day",
                           font=dict(size=12, color=ink("muted")))
        base_layout(fig, 200, f"Item completeness - participant {user_id}, {local_date}")
        return fig

    meta = ["ema_id", "ema_type", "completeness", "missing_b1b2", "item_bank_version"]
    columns = [column for column in matrix.columns if column not in meta]
    state_index = {state: index for index, state in enumerate(COMPLETENESS_STATES)}
    z = [[state_index.get(row[column], 2) for column in columns]
         for row in matrix.to_dict("records")]
    labels = [f"{row['ema_id']} ({row['ema_type'].replace('_', ' ')})"
              for row in matrix.to_dict("records")]

    # answered / not answered are a validated all-pairs pair; not applicable is a
    # neutral, never a third hue - it means the branch closed, not a failure.
    scale = [[0.0, ink("series_1")], [0.33, ink("series_1")],
             [0.34, STATUS["critical"]], [0.66, STATUS["critical"]],
             [0.67, ink("grid")], [1.0, ink("grid")]]
    fig.add_trace(go.Heatmap(
        z=z, x=columns, y=labels, zmin=0, zmax=2, colorscale=scale, showscale=False,
        xgap=2, ygap=2,
        text=[[row[column] for column in columns] for row in matrix.to_dict("records")],
        hovertemplate="%{y}<br>%{x}<br>%{text}<extra></extra>"))

    for column in SIGNAL_SUB_ITEMS:
        if column not in columns:
            continue
        position = columns.index(column)
        fig.add_shape(type="rect", x0=position - 0.5, x1=position + 0.5,
                      y0=-0.5, y1=len(labels) - 0.5,
                      line=dict(color=STATUS["critical"], width=2), fillcolor="rgba(0,0,0,0)")

    for index, state in enumerate(COMPLETENESS_STATES):
        colour = [ink("series_1"), STATUS["critical"], ink("grid")][index]
        fig.add_trace(go.Scatter(
            x=[None], y=[None], mode="markers", name=state,
            marker=dict(size=10, color=colour, line=dict(color=ink("axis"), width=1)),
            showlegend=True))

    missing = int(matrix["missing_b1b2"].fillna(False).sum())
    base_layout(fig, max(320, 46 * len(labels) + 220),
                f"Item completeness - participant {user_id}, {local_date}",
                margin=dict(l=210, r=40, t=104, b=150))
    fig.add_annotation(
        x=1.0, y=1.0, xref="paper", yref="paper", xanchor="right", yanchor="bottom",
        yshift=26, showarrow=False,
        text=(f"<span style='color:{STATUS['critical']}'>{STATUS_ICON['critical']}</span> "
              f"{missing} check-in(s) missing B1/B2 - no decision point produced"
              if missing else
              f"<span style='color:{STATUS['good']}'>{STATUS_ICON['good']}</span> "
              "B1/B2 present on every check-in"),
        font=dict(family=FONT, size=11, color=ink("text")))
    fig.update_layout(plot_bgcolor=ink("page"), showlegend=True,
                      legend=dict(orientation="h", y=1.02, x=0, yanchor="bottom",
                                  font=dict(size=10, color=ink("text_secondary"))))
    fig.update_xaxes(tickangle=-60, tickfont=dict(size=9, color=ink("muted")),
                     title=dict(text="sub-item served that day (frozen item bank v1)",
                                font=dict(size=11, color=ink("muted"))))
    fig.update_yaxes(autorange="reversed", tickfont=dict(size=10, color=ink("text")),
                     title=dict(text="check-in", font=dict(size=11, color=ink("muted"))))
    return fig

In [5]:
# The riskiest participant is often the one with nothing in their lanes, so the
# deep dive picks the busiest instead, and the most recent day that actually has
# both decision points and check-ins to look at.
TIMELINE_USER = int(jitai_log_df["user_id"].value_counts().idxmax())
TIMELINE_DATE = max(
    (day["local_date"] for day in timeline(TIMELINE_USER, days=14)
     if day["decisions"] and day["checkins"]["emas"]),
    default=today_local())
print(f"participant {TIMELINE_USER} on {TIMELINE_DATE}")

participant 430 on 2026-09-11


## 3A - Six-lane daily strip

In [6]:
# One day, six lanes, one shared clock. Top to bottom:
#   Wear      - minute-by-minute presence of a heart-rate sample; the shaded
#               band is the 08:00-22:00 waking window that wear is scored
#               against, and gaps past 2h inside it are filled red.
#   Sync      - how far behind the device's sync clock had fallen. A blank lane
#               means no writer, never a flat line at zero.
#   Check-ins - slot boundaries as dashed rules, submissions coloured by
#               completeness, reminder ticks labelled with their slot index.
#   Decisions - one row per engine outcome, so position carries identity.
#   Delivery  - push to device to receipt as bar length, with the 2h outcome
#               window shaded and the linked check-in marked inside it.
#   MSSD      - observed volatility against the threshold it was compared to.
# On this day participant 1001 wore the watch 815 of 1440 minutes with one gap
# past two hours, synced 3 times, submitted 2 of 6 check-ins after 3 reminders,
# and produced 4 decision points - all four "below within-person threshold", so
# no prompt was sent and the delivery lane is empty.
plot_timeline_day(TIMELINE_USER, TIMELINE_DATE)

## 3B - Delivery funnel

In [7]:
# The whole study for one participant: where prompts are lost between the
# engine deciding to send and the participant doing something.
# For 1001 all 7 sent prompts reached the device and reported a receipt, and 6
# drew an engagement event - a 14% drop at the last step only. A large drop at
# device_received_at instead would point at push infrastructure rather than at
# the participant.
plot_delivery_funnel(TIMELINE_USER)

In [8]:
# Delivery split by platform. iOS and Android push behaviour diverges, and a
# platform-specific collapse is invisible in the pooled funnel above.
# 1001's 7 delivered prompts split 4 iOS / 3 Android - no asymmetry here.
splits = plot_delivery_splits(TIMELINE_USER)
splits["receipt_platform"]

In [9]:
# The same prompts by the app state they arrived in. 5 of 7 landed with the app
# in the background, which is the state that matters for push reliability -
# a foreground-only delivery record would mean the silent push is not waking the
# app as intended.
splits["receipt_app_state"]

## 3C - Item completeness

In [10]:
# One row per check-in that day, one column per sub-item actually served, three
# states: answered, not answered, and not applicable (a branching gate closed,
# which is not a failure).
# B1_valence, B1_arousal and B2_stress are ringed in red because they feed
# calculate_mssd: a check-in missing them produces no decision point at all,
# which is a hole in the trial rather than a data-quality note.
# Both of 1001's check-ins are 100% complete on the served set; the 9
# not-applicable cells are the B4-B7 rotation differing between the two.
plot_completeness_matrix(TIMELINE_USER, TIMELINE_DATE)

/var/folders/6y/w83434vd58jf6rqmf82sh3yc0000gn/T/ipykernel_48051/2131150187.py:46: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  missing = int(matrix["missing_b1b2"].fillna(False).sum())


In [11]:
# A week of lanes at a glance, to pick the day worth opening in full.
pd.DataFrame([
    {"local_date": day["local_date"],
     "wear_minutes": day["wear"]["covered_minutes"],
     "gaps_over_2h": len(day["wear"]["gaps_gt_2h"]),
     "sync_advances": len(day["sync"]["advances"]),
     "check_ins": len(day["checkins"]["emas"]),
     "reminders": len(day["checkins"]["reminders"]),
     "decision_points": len(day["decisions"]),
     "delivered": len(day["delivery"]),
     "unexplained_mssd": sum(1 for point in day["mssd"] if point["unexplained"])}
    for day in timeline(TIMELINE_USER, days=7)])

,local_date,wear_minutes,gaps_over_2h,sync_advances,check_ins,reminders,decision_points,delivered,unexplained_mssd
0,2026-09-12,0,0,0,0,6,0,0,0
1,2026-09-13,0,0,0,0,6,0,0,0
2,2026-09-14,0,0,0,0,6,0,0,0
3,2026-09-15,0,0,0,0,6,0,0,0
4,2026-09-16,0,0,0,0,6,0,0,0
5,2026-09-17,0,0,0,0,6,0,0,0
6,2026-09-18,0,0,0,0,0,0,0,0
